# Lab 5: Build and Train LSTM for Text Classification and Sequence-to-Sequence

This notebook contains two parts:

1. LSTM for text classification using the IMDB sentiment dataset.
2. LSTM Sequence-to-Sequence model for reversing digit sequences.

Run each block in order in Google Colab.

In [ ]:
# ============================================================
# BLOCK 1: Import required libraries
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input

print("TensorFlow version:", tf.__version__)

## Part A: LSTM for Text Classification

The task is binary sentiment classification. The model predicts whether a movie review is positive or negative.

In [ ]:
# ============================================================
# BLOCK 2: Load and preprocess IMDB dataset
# ============================================================

# Use the top 10,000 most frequent words from the dataset.
VOCAB_SIZE = 10000

# Each review is padded or truncated to 200 words.
MAX_LEN = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# Use a smaller subset so the notebook trains quickly in Colab.
x_train = x_train[:12000]
y_train = y_train[:12000]
x_test = x_test[:3000]
y_test = y_test[:3000]

# Padding makes all review sequences the same length.
x_train = pad_sequences(x_train, maxlen=MAX_LEN, padding="post", truncating="post")
x_test = pad_sequences(x_test, maxlen=MAX_LEN, padding="post", truncating="post")

print("Training data shape:", x_train.shape)
print("Testing data shape:", x_test.shape)

In [ ]:
# ============================================================
# BLOCK 3: Build LSTM text classification model
# ============================================================

lstm_classifier = Sequential([
    # Converts word IDs into dense vector representations.
    Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LEN),

    # LSTM learns long-term sequence patterns in text.
    LSTM(128),

    # Dropout reduces overfitting.
    Dropout(0.5),

    Dense(64, activation="relu"),

    # Sigmoid is used because this is binary classification.
    Dense(1, activation="sigmoid")
])

lstm_classifier.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_classifier.summary()

In [ ]:
# ============================================================
# BLOCK 4: Train the LSTM classifier
# ============================================================

classification_history = lstm_classifier.fit(
    x_train,
    y_train,
    epochs=4,
    batch_size=64,
    validation_split=0.2
)

In [ ]:
# ============================================================
# BLOCK 5: Evaluate the LSTM classifier
# ============================================================

test_loss, test_accuracy = lstm_classifier.evaluate(x_test, y_test)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)

In [ ]:
# ============================================================
# BLOCK 6: Plot classification accuracy and loss
# ============================================================

plt.figure(figsize=(8, 5))
plt.plot(classification_history.history["accuracy"], label="Training Accuracy")
plt.plot(classification_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("LSTM Text Classification Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(classification_history.history["loss"], label="Training Loss")
plt.plot(classification_history.history["val_loss"], label="Validation Loss")
plt.title("LSTM Text Classification Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## Part B: LSTM Sequence-to-Sequence Model

This Seq2Seq model learns to reverse a sequence of digits.

Example:

`Input: 1 4 7 2`

`Output: 2 7 4 1`

In [ ]:
# ============================================================
# BLOCK 7: Create toy Seq2Seq dataset
# ============================================================

PAD = 0
START = 1
END = 2

# Digits 0-9 are represented by token IDs 3-12.
DIGIT_OFFSET = 3
SEQ_VOCAB_SIZE = 13
MAX_SEQ_LEN = 8

def generate_seq2seq_data(num_samples=12000):
    encoder_inputs = []
    decoder_inputs = []
    decoder_outputs = []

    for _ in range(num_samples):
        length = np.random.randint(3, MAX_SEQ_LEN + 1)

        digits = np.random.randint(0, 10, size=length)
        source = digits + DIGIT_OFFSET
        target = source[::-1]

        enc = np.zeros(MAX_SEQ_LEN, dtype="int32")
        dec_in = np.zeros(MAX_SEQ_LEN + 1, dtype="int32")
        dec_out = np.zeros(MAX_SEQ_LEN + 1, dtype="int32")

        enc[:length] = source

        # Decoder input starts with START token.
        dec_in[0] = START
        dec_in[1:length + 1] = target

        # Decoder output ends with END token.
        dec_out[:length] = target
        dec_out[length] = END

        encoder_inputs.append(enc)
        decoder_inputs.append(dec_in)
        decoder_outputs.append(dec_out)

    return np.array(encoder_inputs), np.array(decoder_inputs), np.array(decoder_outputs)

encoder_input_data, decoder_input_data, decoder_output_data = generate_seq2seq_data()

print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder output shape:", decoder_output_data.shape)

In [ ]:
# ============================================================
# BLOCK 8: Build LSTM Encoder-Decoder Seq2Seq model
# ============================================================

EMBED_DIM = 64
LATENT_DIM = 128

# Encoder
encoder_inputs = Input(shape=(MAX_SEQ_LEN,), name="encoder_inputs")
encoder_embedding = Embedding(SEQ_VOCAB_SIZE, EMBED_DIM, mask_zero=True, name="encoder_embedding")
encoder_embedded = encoder_embedding(encoder_inputs)

encoder_lstm = LSTM(LATENT_DIM, return_state=True, name="encoder_lstm")
encoder_state_h, encoder_state_c = encoder_lstm(encoder_embedded)[1:]
encoder_states = [encoder_state_h, encoder_state_c]

# Decoder
decoder_inputs = Input(shape=(MAX_SEQ_LEN + 1,), name="decoder_inputs")
decoder_embedding = Embedding(SEQ_VOCAB_SIZE, EMBED_DIM, mask_zero=True, name="decoder_embedding")
decoder_embedded = decoder_embedding(decoder_inputs)

decoder_lstm = LSTM(LATENT_DIM, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_embedded, initial_state=encoder_states)

decoder_dense = Dense(SEQ_VOCAB_SIZE, activation="softmax", name="output_layer")
final_outputs = decoder_dense(decoder_outputs)

seq2seq_model = Model([encoder_inputs, decoder_inputs], final_outputs)

seq2seq_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

seq2seq_model.summary()

In [ ]:
# ============================================================
# BLOCK 9: Train Seq2Seq model
# ============================================================

seq2seq_history = seq2seq_model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_output_data,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

In [ ]:
# ============================================================
# BLOCK 10: Test Seq2Seq model predictions
# ============================================================

def token_to_text(tokens):
    text_tokens = []

    for token in tokens:
        token = int(token)

        if token == PAD:
            continue
        elif token == START:
            text_tokens.append("<start>")
        elif token == END:
            text_tokens.append("<end>")
        else:
            text_tokens.append(str(token - DIGIT_OFFSET))

    return text_tokens

def predict_reverse_sequence(sample_index):
    enc = encoder_input_data[sample_index:sample_index + 1]
    dec = decoder_input_data[sample_index:sample_index + 1]

    predictions = seq2seq_model.predict([enc, dec], verbose=0)
    predicted_tokens = np.argmax(predictions[0], axis=-1)

    print("Input sequence:   ", token_to_text(enc[0]))
    print("Expected output:  ", token_to_text(decoder_output_data[sample_index]))
    print("Predicted output: ", token_to_text(predicted_tokens))

predict_reverse_sequence(0)
predict_reverse_sequence(1)
predict_reverse_sequence(2)

In [ ]:
# ============================================================
# BLOCK 11: Plot Seq2Seq training curves
# ============================================================

plt.figure(figsize=(8, 5))
plt.plot(seq2seq_history.history["accuracy"], label="Training Accuracy")
plt.plot(seq2seq_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Seq2Seq LSTM Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(seq2seq_history.history["loss"], label="Training Loss")
plt.plot(seq2seq_history.history["val_loss"], label="Validation Loss")
plt.title("Seq2Seq LSTM Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## Lab 5 Conclusion

The LSTM text classifier learns sentiment patterns from review sequences. The Seq2Seq LSTM model uses an encoder to understand the input sequence and a decoder to generate the output sequence. LSTM is useful for sequence problems because it can preserve long-term information using memory cells and gates.